In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA

from koopmann.data import DatasetConfig, get_dataset_class
from koopmann.vit import HF_VIT_DICT, get_hf_vision_model

In [2]:
data_config = DatasetConfig(
    dataset_name="ImagenetteDataset",
    num_samples=-1,
    split="train",
    root="/mnt/nishant/datasets/",
)
DatasetClass = get_dataset_class(data_config.dataset_name)
dataset = DatasetClass(config=data_config)
rand_idx = torch.randperm(len(dataset)).tolist()

In [3]:
hf_model, hf_processor = get_hf_vision_model(
    hf_name=HF_VIT_DICT["dinov3_small"],
    cache_dir="/mnt/nishant/huggingface",
    device="cuda",
)
_ = hf_model.eval()

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/86.4M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

In [8]:
batch_size = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hf_model = hf_model.to(device).eval()

num_images_to_process = 1000
images, labels = zip(*[dataset[i] for i in rand_idx[:num_images_to_process]])

token_idx = 0
all_cls = []

for i in range(0, num_images_to_process, batch_size):
    batch_images = list(images[i : i + batch_size])
    inputs = hf_processor(batch_images, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = hf_model(**inputs, output_hidden_states=True)

        # CLS hidden states across layers: (batch, layers, hidden_dim)
        cls_batch = torch.stack(
            [h[:, token_idx] for h in outputs.hidden_states], dim=1
        ).cpu()

    all_cls.append(cls_batch)

# Final tensors
cls_hidden_states = torch.cat(all_cls, dim=0)  # (B, L, D)
labels = np.array(labels)  # keep as numpy for later mapping

In [9]:
# cls_hidden_states: (B, L, D) on CPU
cls_np = cls_hidden_states.numpy()
B, L, D = cls_np.shape

# labels: length-B tuple
labels = np.array(labels)

# ---- 1. Normalize per layer (same as you had) ----
layer_mean = cls_np.mean(axis=0)  # (L, D)
layer_std = cls_np.std(axis=0, ddof=1) + 1e-6  # (L, D)
cls_norm = (cls_np - layer_mean) / layer_std  # (B, L, D)


# ---- 2. Orthogonal Procrustes helper (rotation + translation, no scaling) ----
def align_to_reference(ref, X):
    """
    Align X to ref with an orthogonal Procrustes transform:
    find R s.t. X R ~ ref (minimizing Frobenius norm),
    and add back ref mean.
    ref, X: (B, 3)
    """
    ref_mean = ref.mean(axis=0, keepdims=True)
    X_mean = X.mean(axis=0, keepdims=True)

    ref0 = ref - ref_mean
    X0 = X - X_mean

    # YR ≈ X  => here we want X0 @ R ≈ ref0
    # solve for R via SVD of X0^T ref0
    U, _, VT = np.linalg.svd(X0.T @ ref0, full_matrices=False)
    R = U @ VT  # (3, 3), rotation

    X_aligned = X0 @ R + ref_mean  # (B, 3)
    return X_aligned


# ---- 3. Choose a reference layer and compute its 3D PCA ----
align_idx = L - 1  # e.g. last layer as reference
pca_ref = PCA(n_components=3)
ref_3d = pca_ref.fit_transform(cls_norm[:, align_idx, :])  # (B, 3)

# ---- 4. For each layer: PCA -> align to reference ----
coords_aligned = np.zeros((B, L, 3), dtype=np.float32)

for t in range(L):
    pca_t = PCA(n_components=3)
    X_t_3d = pca_t.fit_transform(cls_norm[:, t, :])  # (B, 3)
    X_t_aligned = align_to_reference(ref_3d, X_t_3d)  # (B, 3)
    coords_aligned[:, t, :] = X_t_aligned

# ---- 5. Build DataFrame ----
coords_flat = coords_aligned.reshape(B * L, 3)

df = pd.DataFrame(
    {
        "pc1": coords_flat[:, 0],
        "pc2": coords_flat[:, 1],
        "pc3": coords_flat[:, 2],
        "layer": np.tile(np.arange(L), B),
        "sample": np.repeat(np.arange(B), L),
        "label": np.repeat(labels, L).astype(str),
    }
)

# ---- 6. Fixed axes (global min/max over all layers) ----
pad = 0.05


def padded_range(col, pad=0.05):
    vmin, vmax = col.min(), col.max()
    span = vmax - vmin
    if span == 0:
        span = 1.0
    return [vmin - pad * span, vmax + pad * span]


x_range = padded_range(df["pc1"], pad)
y_range = padded_range(df["pc2"], pad)
z_range = padded_range(df["pc3"], pad)

# ---- 7. Plot: PCA per layer + Procrustes alignment, animated over layers ----
fig = px.scatter_3d(
    df,
    x="pc1",
    y="pc2",
    z="pc3",
    animation_frame="layer",
    color="label",
    hover_data=["sample", "label"],
)

fig.update_layout(
    scene=dict(
        xaxis=dict(range=x_range),
        yaxis=dict(range=y_range),
        zaxis=dict(range=z_range),
        aspectmode="cube",
    )
)
fig.update_traces(marker_size=4)

fig.show()


In [10]:
# cls_hidden_states: (B, L, D) tensor
X = cls_hidden_states.to(device)  # (B, L, D)

# labels: numpy array of shape (B,)
y_local = torch.from_numpy(labels).long().to(device)  # (B,)

B, L, D = X.shape
num_classes = int(y_local.max().item()) + 1

epochs = 1_000
lr = 1e-3

acc_per_layer = []

# ---------------------------
# 1. Linear probes per layer
# ---------------------------
for l in range(L):
    # CLS representation at layer l: (B, D)
    X_l = X[:, l, :]  # (B, D)

    probe = nn.Linear(D, num_classes).to(device)
    optimizer = optim.Adam(probe.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # full-batch training loop
    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = probe(X_l)  # (B, num_classes)
        loss = criterion(logits, y_local)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        logits = probe(X_l)
        preds = logits.argmax(dim=-1)
        acc = (preds == y_local).float().mean().item()

    acc_per_layer.append(acc)
    print(f"Probe layer {l}: train acc = {acc:.4f}")


Probe layer 0: train acc = 0.1110
Probe layer 1: train acc = 0.2900
Probe layer 2: train acc = 0.6160
Probe layer 3: train acc = 0.6670
Probe layer 4: train acc = 0.7680
Probe layer 5: train acc = 0.8830
Probe layer 6: train acc = 0.9630
Probe layer 7: train acc = 0.9970
Probe layer 8: train acc = 0.9860
Probe layer 9: train acc = 0.9990
Probe layer 10: train acc = 1.0000
Probe layer 11: train acc = 1.0000
Probe layer 12: train acc = 1.0000
